In [1]:
import logging
import warnings

import matplotlib.pyplot as plt
import mlflow
import mlflow.data
import numpy as np
import pandas as pd
from mlflow.client import MlflowClient
from mlflow.data.pandas_dataset import PandasDataset
from utilsforecast.plotting import plot_series

from sklearn.metrics import mean_absolute_error, root_mean_squared_error

from neuralforecast.core import NeuralForecast
from neuralforecast.models import NBEATSx
from neuralforecast.utils import AirPassengersDF
from neuralforecast.losses.pytorch import MAE

In [2]:
logging.getLogger("mlflow").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

In [3]:
from pathlib import Path

# Resolve project root dynamically
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

db_path = project_root / "mlflow.db"
artifact_path = project_root / "mlartifacts"

# Set tracking URI to SQLite
mlflow.set_tracking_uri(f"sqlite:///{db_path.as_posix()}")

# Create experiment with explicit artifact destination
experiment_name = "quickstart_nixtla"
client = MlflowClient()

experiment = client.get_experiment_by_name(experiment_name)
if experiment is None:
    client.create_experiment(
        name=experiment_name,
        artifact_location=artifact_path.as_uri()
    )

mlflow.set_experiment(experiment_name)

<Experiment: artifact_location='file:///home/on1link/Applications/projects/store_sales/store_sales_forecasting/mlartifacts', creation_time=1787335185332, effective_trace_archival_retention=None, experiment_id='5', last_update_time=1787335185332, lifecycle_stage='active', name='quickstart_nixtla', tags={}, trace_location=None, workspace='default'>

In [4]:
# Split data and declare panel dataset
Y_df = AirPassengersDF
Y_train_df = Y_df[Y_df.ds<='1959-12-31'] # 132 train
Y_test_df = Y_df[Y_df.ds>'1959-12-31'] # 12 test
Y_df.tail()

,unique_id,ds,y
139,1.0,1960-08-31,606.0
140,1.0,1960-09-30,508.0
141,1.0,1960-10-31,461.0
142,1.0,1960-11-30,390.0
143,1.0,1960-12-31,432.0


In [11]:
import torch


mlflow.pytorch.autolog(checkpoint=False)

with mlflow.start_run() as run:
    # Log the dataset to the MLflow Run. Specify the "training" context to indicate that the
    # dataset is used for model training
    dataset: PandasDataset = mlflow.data.from_pandas(Y_df, source="AirPassengersDF")
    mlflow.log_input(dataset, context="training")

    # Define and log parameters
    horizon = len(Y_test_df)
    model_params = dict(
        input_size=1 * horizon,
        h=horizon,
        max_steps=300,  
        loss=MAE(),
        valid_loss=MAE(),  
        activation='ReLU',
        scaler_type='robust',
        random_seed=42,
        enable_progress_bar=False,
    )
    mlflow.log_params(model_params)

    # Fit NBEATSx model
    models = [NBEATSx(**model_params)]
    nf = NeuralForecast(models=models, freq='M')           
    train = nf.fit(df=Y_train_df, val_size=horizon)
    
    # Save conda environment used to run the model (if you used a conda environment)
    mlflow.pytorch.get_default_conda_env()

    # Save pip requirements
    mlflow.pytorch.get_default_pip_requirements()

    sample_input = torch.randn(1, 1, model_params["input_size"])
    mlflow.pytorch.log_model(
        pytorch_model=nf.models[0],
        artifact_path="model",
        serialization_format="pickle",
        input_example=sample_input,
        registered_model_name="NBEATSx_AirPassengers"
    )

    Y_hat_df = nf.predict(futr_df=Y_test_df)
    mlflow.log_metrics({'mae': mean_absolute_error(Y_test_df['y'], Y_hat_df['NBEATSx']),
                    'rmse': root_mean_squared_error(Y_test_df['y'], Y_hat_df['NBEATSx'])}, dataset=dataset)
    print(f"Run completed. Run ID: {run.info.run_id}")

mlflow.pytorch.autolog(disable=True)

# Save the neural forecast model
nf.save(path='./checkpoints/test_run_1/',
        model_index=None, 
        overwrite=True,
        save_dataset=True)

Seed set to 42
Successfully registered model 'NBEATSx_AirPassengers'.
Created version '1' of model 'NBEATSx_AirPassengers'.


Run completed. Run ID: 610379ec929a42a383e8eccb0a77e55d


In [16]:
# Register model
model_name = "NBEATSx_AirPassengers"

# Get the latest registered version for this model
latest_version = client.get_latest_versions(model_name, stages=["None"])[0].version

# Assign the alias 'monk' to the registered version
client.set_registered_model_alias(
    name=model_name,
    alias="monk",
    version=latest_version
)

print(f"Assigned alias 'monk' to model '{model_name}' (Version {latest_version})")

Assigned alias 'monk' to model 'NBEATSx_AirPassengers' (Version 1)


In [ ]:
plot_series(Y_train_df, Y_hat_df, palette='tab20b')

In [10]:
Y_test_df.tail(1)

,unique_id,ds,y
143,1.0,1960-12-31,432.0
